# Fourth Notebook to use
Data only missing encoding which I will do in here instead, though realistically it matters little where it is done.

In [18]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import ast

I want to calculate the average of all my personal ratings ive given to each movie of a specific director, to have some way of meassure how much I like each director.

So I need to calculate a feature after splitting to training, validation and test sets. The standard way to do this such that it's even respected in a cv fold is through a Pipeline. 

Just like with using PolynomialFeatures in a pipeline, it's the same idea here! So first define an encoder.

In [19]:
class DirectorTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y):    # this is ONLY called during training!
        df = X.copy()
        df['_target'] = y

        self.global_mean = y.mean()

        stats = (
            df.melt(id_vars="_target", value_vars=["director1", "director2"])
            .dropna()
            .groupby("value")["_target"]
            .mean()
        )

        self.mapping_ = stats.to_dict()
        return self
    
    def transform(self, X): # this is called during training, validation and unseen data!
        avg_avg_personal_scores = []
        for d1, d2 in zip(X['director1'], X['director2']):
            s1 = self.mapping_.get(d1, self.global_mean)
            s2 = self.mapping_.get(d2, self.global_mean) if d2 else None

            avg_avg_personal_scores.append(s1 if not s2 else (s1 + s2)/2)
        
        return pd.DataFrame({'avg_personal_score': avg_avg_personal_scores}, index=X.index)

        

In [20]:
preprocess = ColumnTransformer(
    transformers=[
        ("director", DirectorTargetEncoder(), ["director1", "director2"]),
    ],
    remainder="passthrough" # all other columns are automatically included
) 

# Make training and unseen data correct format! So finalize presentation of datasets

In [21]:
df = pd.read_csv('all_data.csv')
df.head(2)

,title,release_year,personal_rating,avg_rating,genre,country,language,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies,director1,director2
0,2 Fast 2 Furious,2003,2.5,3.15,"['Crime', 'Action', 'Thriller']","['Germany', 'USA']","['English', 'English', 'Spanish']",12.0,3.358417,3.0871,john singleton,NaN
1,2012,2009,2.0,2.55,"['Adventure', 'Action', 'Science Fiction']",['USA'],"['English', 'Russian', 'Hindi', 'German', 'Ita...",19.0,3.072658,3.1191,roland emmerich,NaN


In [22]:
df_unseen = pd.read_csv('unseen_movies.csv')
df_unseen.head(2)

,title,release_year,avg_rating,genre,director,country,language
0,schindler's list,1993,4.2825,"['Drama', 'History', 'War']",['Steven Spielberg'],['US'],"['German', 'Polish', 'Hebrew', 'English']"
1,12 angry men,1957,4.3000,['Drama'],['Sidney Lumet'],['US'],['English']


Make all categorical values to actual list of strings first.

In [14]:
# need to convert the string of list of strings to just a list of strings to do encoding
cat_features = ['genre', 'country', 'language']

for column in cat_features:
    df[column] = df[column].apply(lambda x: ast.literal_eval(x))
    df_unseen[column] = df_unseen[column].apply(lambda x: ast.literal_eval(x))

In [17]:
for directors in df_unseen['director']:
    print(directors)

['Steven Spielberg']
['Sidney Lumet', 'Don Kranze']
['Hayao Miyazaki', 'Atsushi Takahashi', 'Masayuki Miyaji']
['Aditya Chopra']
['Frank Darabont']
['Kim Seong-sik', 'Lee Jung-hoon', 'Bong Joon Ho']
['Quentin Tarantino']
['Makoto Shinkai', 'Kenji Imura', 'Yoshitoshi Shinomiya']
['Sergio Leone', 'Fabrizio Gianni', 'Giancarlo Santi']
['Akira Kurosawa', 'Yasuyoshi Tajitsu', 'Sakae Hirosawa', 'Hiromichi Horikawa', 'Toshi Kaneko', 'Masaya Shimizu']
['Martin Scorsese', 'Joseph P. Reidy']
['Isao Takahata']
['Roberto Benigni']
['Masaki Kobayashi', 'Kôji Niwa']
['Giuseppe Tornatore', 'Giuseppe Giglietti']
['Fernando Meirelles', 'Kátia Lund']
['Ernesto Contreras']
['Guel Arraes', 'Paola Pol Balloussier', 'Flávia Lacerda']
['Alfred Hitchcock', 'Hilton A. Green']
['Miloš Forman', 'Irby Smith']
['Tosca Musk']
['Erik de Bruyn']
['Sergio Leone', 'Luca Morsella', 'Amy Wells', 'Philippe Landoulsi', 'Dennis T. Benatar']
['Hayao Miyazaki', 'Ryosuke Kiyokawa', 'Yosuke Toba']
['Naoko Yamada', 'Eisaku Kawan

In [ ]:
# from useful_funcs import normalize_country
# df['country'] = df['country'].apply(normalize_country)

In [21]:
df.melt(id_vars="personal_rating", value_vars=["director1", "director2"])

,personal_rating,variable,value
0,2.5,director1,john singleton
1,2.0,director1,roland emmerich
2,0.5,director1,sam mendes
3,3.5,director1,mary harron
4,4.0,director1,thomas vinterberg
...,...,...,...
225,2.5,director2,No one
226,3.5,director2,No one
227,3.5,director2,No one
228,3.5,director2,No one


In [52]:
for director in df['director1']:
    pass

In [4]:
unique_directors = pd.concat([df["director1"], df["director2"]]).dropna().unique()
unique_directors = [d for d in unique_directors if d != 'No one']

In [6]:
avg_personal_scores = []
nums = []
for director in unique_directors:
    mean = df[(df['director1'] == director) | (df['director2'] == director)]['personal_rating'].mean()
    num = len(df[(df['director1'] == director) | (df['director2'] == director)])
    avg_personal_scores.append(mean)
    nums.append(num)

In [9]:
df2 = pd.DataFrame({
    'avg_personal_score': avg_personal_scores
},
index=unique_directors)

In [19]:
avg_avg_personal_scores = []
for d1, d2 in zip(df['director1'], df['director2']):
        if d2 == 'No one':
                avg_avg_personal_scores.append(df2['avg_personal_score'].get(d1))
        else:
                avg_avg_personal_scores.append((df2['avg_personal_score'].get(d1)+df2['avg_personal_score'].get(d2))/2)
        

In [61]:
df[(df['director1'] == 'joe russo') | (df['director2'] == 'joe russo')]['personal_rating'].mean()

np.float64(4.0)

In [45]:
df2 = df.groupby('director1')['personal_rating'].agg(mean_rating = 'mean', num = 'count').reset_index()
df2.sort_values('num', ascending=False)

,director1,mean_rating,num
13,christopher nolan,3.90,10
27,james cameron,3.10,5
11,chris columbus,2.00,4
51,peter jackson,2.25,4
46,michael bay,2.75,4
...,...,...,...
67,sunghoo park,4.00,1
68,thomas vinterberg,4.00,1
69,tim burton,2.00,1
70,tim miller,3.50,1


In [41]:
mean_map = df2.set_index('director')['mean_rating']
df['avg_personal_director_rating'] = df['director'].map(mean_map)
df.sort_values('avg_personal_director_rating', ascending=False).head()

,title,release_year,personal_rating,avg_rating,genre,director,country,language,director_num_movies,director_avg_movie_score,director_avg_last_5_movies,avg_personal_rating,avg_personal_director_rating
9,Avengers: Infinity War,2018,5.0,4.02,"['Action', 'Science Fiction', 'Adventure']",anthony russo,['USA'],"['English', 'English', 'Xhosa']",10.0,3.351050,3.7192,5.0,5.0
89,The Intouchables,2011,5.0,4.15,"['Drama', 'Comedy']",olivier nakache,['France'],"['French', 'English', 'French']",10.0,3.320500,3.5357,5.0,5.0
64,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"['Animation', 'Science Fiction', 'Adventure', ...",kemp powers,['USA'],"['English', 'English', 'Hindi', 'Italian', 'Sp...",1.5,4.139500,4.1395,5.0,5.0
100,The Shawshank Redemption,1994,5.0,4.58,"['Crime', 'Drama']",frank darabont,['USA'],['English'],7.0,3.597714,3.8368,5.0,5.0
65,Spider-Man: Into the Spider-Verse,2018,4.5,4.40,"['Adventure', 'Animation', 'Action', 'Science ...",bob persichetti,['USA'],"['English', 'English', 'Japanese', 'Spanish']",3.5,3.733792,3.7785,4.5,4.5


In [43]:
df[df['director'] == 'anthony russo']

,title,release_year,personal_rating,avg_rating,genre,director,country,language,director_num_movies,director_avg_movie_score,director_avg_last_5_movies,avg_personal_rating,avg_personal_director_rating
9,Avengers: Infinity War,2018,5.0,4.02,"['Action', 'Science Fiction', 'Adventure']",anthony russo,['USA'],"['English', 'English', 'Xhosa']",10.0,3.35105,3.7192,5.0,5.0
